# 03. 狀態管理

深入了解 LangGraph 的狀態管理機制。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 設計適合需求的狀態結構
- ✅ 理解並使用不同的 Reducer
- ✅ 實作自訂 Reducer
- ✅ 處理複雜的多欄位狀態

---

## 📊 狀態更新機制

```
┌─────────────────────────────────────────────────────────┐
│                    狀態更新流程                          │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────┐                       ┌─────────┐        │
│   │ 節點 A  │ ── return {x: 1} ──▶ │ Reducer │        │
│   └─────────┘                       └────┬────┘        │
│                                          │             │
│   ┌─────────┐                       ┌────▼────┐        │
│   │ 節點 B  │ ── return {x: 2} ──▶ │  State  │        │
│   └─────────┘                       └─────────┘        │
│                                                         │
│   沒有 Reducer: x = 2 (覆蓋)                            │
│   add Reducer: x = [1, 2] (累加)                        │
│   max Reducer: x = 2 (取最大)                           │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

In [1]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

---

## 3.1 基本狀態（無 Reducer）

預設行為：**新值覆蓋舊值**

In [2]:
class BasicState(TypedDict):
    """基本狀態 - 直接覆蓋"""
    value: int
    name: str

def double_value(state: BasicState) -> dict:
    """將 value 乘以 2"""
    old = state["value"]
    new = old * 2
    print(f"  double_value: {old} → {new}")
    return {"value": new}

# 建構並測試
graph = StateGraph(BasicState)
graph.add_node("double", double_value)
graph.add_edge(START, "double")
graph.add_edge("double", END)
app = graph.compile()

print("📊 測試基本狀態（覆蓋模式）：")
print("-" * 40)
result = app.invoke({"value": 5, "name": "test"})
print("-" * 40)
print(f"結果: {result}")

📊 測試基本狀態（覆蓋模式）：
----------------------------------------
  double_value: 5 → 10
----------------------------------------
結果: {'value': 10, 'name': 'test'}


---

## 3.2 使用 Reducer 累加

### 內建 Reducer

| Reducer | 來源 | 行為 |
|---------|------|------|
| `add` | `operator` | 列表相加 `[1] + [2] = [1, 2]` |
| `add_messages` | `langgraph` | 智能合併訊息 |

In [3]:
class AccumulatorState(TypedDict):
    """累加狀態 - 使用 add reducer"""
    items: Annotated[list[str], add]  # ← 使用 add reducer
    count: int

def add_item_a(state: AccumulatorState) -> dict:
    print("  add_item_a: 添加 'item_a'")
    return {"items": ["item_a"], "count": state["count"] + 1}

def add_item_b(state: AccumulatorState) -> dict:
    print("  add_item_b: 添加 'item_b'")
    return {"items": ["item_b"], "count": state["count"] + 1}

# 建構圖：A → B
graph = StateGraph(AccumulatorState)
graph.add_node("node_a", add_item_a)
graph.add_node("node_b", add_item_b)
graph.add_edge(START, "node_a")
graph.add_edge("node_a", "node_b")
graph.add_edge("node_b", END)
app = graph.compile()

print("📊 測試 Reducer 累加：")
print("-" * 40)
result = app.invoke({"items": [], "count": 0})
print("-" * 40)
print(f"items: {result['items']}  ← 兩個節點的結果被累加！")
print(f"count: {result['count']}  ← 但 count 被覆蓋（沒有 reducer）")

📊 測試 Reducer 累加：
----------------------------------------
  add_item_a: 添加 'item_a'
  add_item_b: 添加 'item_b'
----------------------------------------
items: ['item_a', 'item_b']  ← 兩個節點的結果被累加！
count: 2  ← 但 count 被覆蓋（沒有 reducer）


---

## 3.3 自訂 Reducer

Reducer 函數簽名：

```python
def my_reducer(current_value, new_value) -> combined_value:
    # current_value: 目前狀態中的值
    # new_value: 節點返回的新值
    return combined_value
```

In [4]:
def max_reducer(current: int, update: int) -> int:
    """只保留最大值的 Reducer"""
    result = max(current, update)
    print(f"    max_reducer: max({current}, {update}) = {result}")
    return result

class MaxState(TypedDict):
    max_score: Annotated[int, max_reducer]  # ← 自訂 reducer

def generate_score(state: MaxState) -> dict:
    import random
    score = random.randint(1, 100)
    print(f"  生成隨機分數: {score}")
    return {"max_score": score}

# 建構：連續執行 3 次
graph = StateGraph(MaxState)
graph.add_node("score1", generate_score)
graph.add_node("score2", generate_score)
graph.add_node("score3", generate_score)
graph.add_edge(START, "score1")
graph.add_edge("score1", "score2")
graph.add_edge("score2", "score3")
graph.add_edge("score3", END)
app = graph.compile()

print("📊 測試自訂 Reducer（取最大值）：")
print("-" * 40)
result = app.invoke({"max_score": 0})
print("-" * 40)
print(f"最大分數: {result['max_score']}")

📊 測試自訂 Reducer（取最大值）：
----------------------------------------
    max_reducer: max(0, 0) = 0
  生成隨機分數: 56
    max_reducer: max(0, 56) = 56
  生成隨機分數: 70
    max_reducer: max(56, 70) = 70
  生成隨機分數: 65
    max_reducer: max(70, 65) = 70
----------------------------------------
最大分數: 70


---

## 3.4 更多自訂 Reducer 範例

In [5]:
# Reducer 1: 保留最近 N 項
def keep_last_n(n: int):
    """工廠函數：建立保留最近 N 項的 reducer"""
    def reducer(current: list, update: list) -> list:
        combined = current + update
        return combined[-n:]  # 只保留最後 n 項
    return reducer

# Reducer 2: 計數器
def counter_reducer(current: int, update: int) -> int:
    """累加計數"""
    return current + update

# Reducer 3: 集合（去重）
def set_reducer(current: set, update: set) -> set:
    """合併集合"""
    return current | update

print("📚 自訂 Reducer 範例：")
print("-" * 40)
print("1. keep_last_n(3): 只保留最近 3 項")
print("2. counter_reducer: 累加計數")
print("3. set_reducer: 合併集合（自動去重）")

📚 自訂 Reducer 範例：
----------------------------------------
1. keep_last_n(3): 只保留最近 3 項
2. counter_reducer: 累加計數
3. set_reducer: 合併集合（自動去重）


---

## 3.5 多欄位狀態

實際應用中常需要追蹤多個欄位：

In [6]:
class MultiFieldState(TypedDict):
    """多欄位狀態範例"""
    messages: Annotated[list, add]  # 累加
    step_count: int                  # 覆蓋
    visited_nodes: Annotated[list, add]  # 累加
    last_node: str                   # 覆蓋

def process_a(state: MultiFieldState) -> dict:
    print("  執行節點 A")
    return {
        "messages": ["Processed by A"],
        "step_count": state["step_count"] + 1,
        "visited_nodes": ["A"],
        "last_node": "A"
    }

def process_b(state: MultiFieldState) -> dict:
    print("  執行節點 B")
    return {
        "messages": ["Processed by B"],
        "step_count": state["step_count"] + 1,
        "visited_nodes": ["B"],
        "last_node": "B"
    }

graph = StateGraph(MultiFieldState)
graph.add_node("a", process_a)
graph.add_node("b", process_b)
graph.add_edge(START, "a")
graph.add_edge("a", "b")
graph.add_edge("b", END)
app = graph.compile()

print("📊 測試多欄位狀態：")
print("-" * 40)
result = app.invoke({
    "messages": ["Start"],
    "step_count": 0,
    "visited_nodes": [],
    "last_node": ""
})
print("-" * 40)
print(f"訊息 (累加): {result['messages']}")
print(f"步數 (覆蓋): {result['step_count']}")
print(f"訪問節點 (累加): {result['visited_nodes']}")
print(f"最後節點 (覆蓋): {result['last_node']}")

📊 測試多欄位狀態：
----------------------------------------
  執行節點 A
  執行節點 B
----------------------------------------
訊息 (累加): ['Start', 'Processed by A', 'Processed by B']
步數 (覆蓋): 2
訪問節點 (累加): ['A', 'B']
最後節點 (覆蓋): B


---

## 💡 重點回顧

### Reducer 類型比較

| 類型 | 語法 | 行為 | 使用場景 |
|------|------|------|----------|
| 無 Reducer | `value: int` | 覆蓋 | 單一值 |
| `add` | `Annotated[list, add]` | 列表相加 | 歷史記錄 |
| `add_messages` | `Annotated[list, add_messages]` | 智能合併 | 對話 |
| 自訂 | `Annotated[T, fn]` | 自訂邏輯 | 特殊需求 |

### 狀態設計原則

```python
class GoodState(TypedDict):
    # ✅ 好的設計
    messages: Annotated[list, add_messages]  # 明確的 reducer
    current_step: str                        # 簡單類型
    error: str | None                        # 可選欄位

class BadState(TypedDict):
    # ❌ 不好的設計
    data: dict        # 太寬泛
    temp: str         # 暫存值不應放狀態
    everything: list  # 命名不清
```

---

## 📝 練習題

1. **實作 min_reducer**：只保留最小值
2. **實作 average_reducer**：計算平均值
3. **設計追蹤狀態**：追蹤每個節點的執行時間
4. **合併策略**：設計一個 reducer 可以處理衝突的更新

---

下一步：[04. ReAct Agent](04_react_agent.ipynb)